# Semana 1: carga, transformacion y reporte del dataset Iris

Este notebook carga el dataset publico `scikit-learn/iris` desde Hugging Face y coordina las clases del proyecto. A partir de las medidas de sepalos y petalos se aplican cuatro transformaciones: renombrar columnas, filtrar registros, crear una caracteristica derivada y seleccionar columnas. El CSV y el reporte `.txt` corresponden al DataFrame transformado.

Ejecuta las celdas en orden. Este dataset publico no requiere autenticacion.

## 1. Preparar el entorno e importar las clases

Se busca la raiz del repositorio para que el notebook funcione al iniciarlo desde la raiz o desde una carpeta interna.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "clases_po").is_dir():
    raise FileNotFoundError("No se encontro src/clases_po desde el directorio actual.")

sys.path.insert(0, str(ROOT / "src"))
from clases_po import HFDatasetLoader, Reportes

DATA_PATH = ROOT / "src" / "sincronicas" / "datos" / "iris_transformado.csv"
REPORT_PATH = ROOT / "src" / "sincronicas" / "reportes" / "iris_transformado.txt"


## 2. Configurar y cargar el dataset desde Hugging Face

`HFDatasetLoader` recibe el identificador del dataset y la division `train`; `load()` devuelve un DataFrame de pandas.

In [2]:
loader = HFDatasetLoader(
    dataset_name="scikit-learn/iris",
    split="train",
)

df_original = loader.load()
print(f"Dataset: {loader.info()['dataset']}")
print(f"Dimensiones originales: {df_original.shape[0]} filas y {df_original.shape[1]} columnas")
df_original.head()

Dataset: scikit-learn/iris
Dimensiones originales: 150 filas y 6 columnas


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 3. Transformacion 1: renombrar las columnas

Se cambian los nombres originales por etiquetas en `snake_case`, mas comodas para trabajar en Python.

In [3]:
df_renombrado = df_original.rename(columns={
    "Id": "id",
    "SepalLengthCm": "sepal_length_cm",
    "SepalWidthCm": "sepal_width_cm",
    "PetalLengthCm": "petal_length_cm",
    "PetalWidthCm": "petal_width_cm",
    "Species": "species",
})

print(df_renombrado.columns.tolist())
df_renombrado.head()

['id', 'sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm', 'species']


,id,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 4. Transformacion 2: filtrar registros

Se conservan las flores con un ancho de sepalo de al menos 2.5 cm. Se muestran las dimensiones antes y despues para observar el efecto del filtro.

In [4]:
df_filtrado = df_renombrado.loc[df_renombrado["sepal_width_cm"] >= 2.5].copy()
print(f"Filas antes del filtro: {len(df_renombrado)}")
print(f"Filas despues del filtro: {len(df_filtrado)}")
df_filtrado.head()

Filas antes del filtro: 150
Filas despues del filtro: 139


,id,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 5. Transformacion 3: crear una caracteristica

Se calcula un area aproximada del petalo multiplicando su largo por su ancho (cm2).

In [5]:
df_con_area = df_filtrado.copy()
df_con_area["petal_area_cm2"] = df_con_area["petal_length_cm"] * df_con_area["petal_width_cm"]
df_con_area[["petal_length_cm", "petal_width_cm", "petal_area_cm2"]].head()

,petal_length_cm,petal_width_cm,petal_area_cm2
0,1.4,0.2,0.28
1,1.4,0.2,0.28
2,1.3,0.2,0.26
3,1.5,0.2,0.30
4,1.4,0.2,0.28


## 6. Transformacion 4: seleccionar columnas

Se conservan las medidas relevantes para observar los petalos, el area calculada y la especie. Este `df_transformado` es el que se guarda y se entrega a la clase de reportes.

In [6]:
columnas_finales = [
    "sepal_length_cm",
    "petal_length_cm",
    "petal_width_cm",
    "petal_area_cm2",
    "species",
]
df_transformado = df_con_area[columnas_finales].copy()
print(f"Dimensiones transformadas: {df_transformado.shape[0]} filas y {df_transformado.shape[1]} columnas")
df_transformado.head()

Dimensiones transformadas: 139 filas y 5 columnas


,sepal_length_cm,petal_length_cm,petal_width_cm,petal_area_cm2,species
0,5.1,1.4,0.2,0.28,Iris-setosa
1,4.9,1.4,0.2,0.28,Iris-setosa
2,4.7,1.3,0.2,0.26,Iris-setosa
3,4.6,1.5,0.2,0.30,Iris-setosa
4,5.0,1.4,0.2,0.28,Iris-setosa


## 7. Generar el reporte y guardar los resultados

La clase `Reportes` recibe el DataFrame transformado para generar el resumen `.txt`. Tambien se exporta ese mismo DataFrame a CSV.

In [7]:
reportes = Reportes()
ruta_reporte = reportes.guardar_reporte(df_transformado, REPORT_PATH)
df_transformado.to_csv(DATA_PATH, index=False)

print(f"CSV transformado guardado en: {DATA_PATH.relative_to(ROOT)}")
print(f"Reporte guardado en: {ruta_reporte.relative_to(ROOT)}")
print("\n" + reportes.generar_reporte(df_transformado))

CSV transformado guardado en: src\sincronicas\datos\iris_transformado.csv
Reporte guardado en: src\sincronicas\reportes\iris_transformado.txt

REPORTE DEL DATAFRAME
Filas: 139
Columnas: 5
Duplicados: 6

Columnas y tipos:
- sepal_length_cm: float64
- petal_length_cm: float64
- petal_width_cm: float64
- petal_area_cm2: float64
- species: object

Valores nulos por columna:
- sepal_length_cm: 0
- petal_length_cm: 0
- petal_width_cm: 0
- petal_area_cm2: 0
- species: 0

Estadísticas numéricas:
       sepal_length_cm  petal_length_cm  petal_width_cm  petal_area_cm2
count          139.000          139.000         139.000         139.000
mean             5.871            3.763           1.207           5.910
std              0.839            1.815           0.787           4.851
min              4.300            1.000           0.100           0.110
25%              5.100            1.500           0.300           0.405
50%              5.800            4.400           1.400           5.980
75%